# SASRec × NSGA-II の実験結果
保存済みの結果を確認するNotebook。再学習は行わない。SASRec、加重和(r,d)、NSGA-II(r,d)を比較する。`r × ジャンル距離`は補助診断値で、最適化には使わない。

In [ ]:
from pathlib import Path
import json
import os
import pandas as pd
from IPython.display import display, Image
project = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'experiment.py').is_file())
run = project / Path(os.environ.get('SERENDIPITY_RUN', 'outputs/full'))
assert (run / 'completion.json').exists(), '先にexperiment.pyを完走させてください'
assert 'genre_distance' in pd.read_csv(run / 'test/pareto.csv', nrows=0).columns, 'SERENDIPITY_RUNに新しい(r,d)の出力を指定してください'
display(json.loads((run / 'completion.json').read_text()))
display(json.loads((run / 'checkpoint.json').read_text()))
display(json.loads((run / 'split.json').read_text()))

## 同じ候補集合で比較
NSGA-IIは3 search seedをユーザー内で平均している。`*_selected`は加重和の重み掃引から、NSGA-IIと同じ予測関連度95%条件の下で平均genre distanceが最大の代表解。test正例による選択ではない。

In [ ]:
summary = pd.read_csv(run / 'test/summary.csv').set_index('method')
display(summary.loc[['sasrec', 'weighted_distance_selected', 'nsga2']])
display(pd.read_csv(run / 'test/paired_bootstrap.csv'))

In [ ]:
display(Image(filename=str(run / 'comparison.png')))
display(pd.read_csv(run / 'training.csv'))

## 推薦されたアイテムを確認
履歴除外・元アイテムID・候補の目的値を追える。候補外の正例も評価の分母に残している。

In [ ]:
recs = pd.read_csv(run / 'test/recommendations.csv')
candidates = pd.read_csv(run / 'test/candidates.csv')
user = recs.user_id.min()
selected = recs[(recs.user_id == user) & recs.method.isin(['sasrec', 'weighted_distance_selected', 'nsga2'])]
display(selected.merge(candidates, on=['user_id', 'item_id']).head(60))

## 判断するときの注意
- 代理目的の改善とheld-out NDCGの変化を分けて読む。
- 関連度95%の制約はNDCG95%維持の保証ではない。
- MovieLensの既存ユーザーに対する接続検証であり、音楽への効果・Fortuitousness・実際の興味は未測定。
- NSGA-IIと加重和で結果が同じなら、この設定では探索コストを正当化する根拠がない。

## 推薦リスト長10、15、20、25、30の比較
SASRecモデルはvalidation NDCG@10で選択したものを共通に使う。候補100件と評価ユーザーを固定し、各Kで再ランキングと評価を行う。Recallは長いリストほど上がりやすいため、同じKの手法間比較も読む。
データ出典：GroupLens MovieLens 100K、Harper and Konstan (2015), The MovieLens Datasets: History and Context, https://doi.org/10.1145/2827872 。利用条件はプロジェクトのDATA_LICENSE.mdを参照。

In [ ]:
sweep = project / Path(os.environ.get('SERENDIPITY_LENGTHS_RUN', 'outputs/list-lengths'))
if (sweep / 'completion.json').exists() and 'genre_distance' in pd.read_csv(sweep / 'k10/test/pareto.csv', nrows=0).columns:
    display(json.loads((sweep / 'completion.json').read_text()))
    lengths = pd.read_csv(sweep / 'summary.csv')
    selected = lengths[(lengths.split == 'test') & lengths.method.isin(['sasrec', 'weighted_distance_selected', 'nsga2'])]
    display(selected[['top_k', 'method', 'recall', 'ndcg', 'relevance', 'genre_distance', 'heldout_distance', 'seconds']])
    display(pd.read_csv(sweep / 'paired_bootstrap.csv').query("split == 'test' and method == 'nsga2' and reference == 'sasrec'"))
    display(pd.read_csv(sweep / 'agreement.csv').query("split == 'test'"))
    display(Image(filename=str(sweep / 'comparison.png')))
else:
    print('新しい(r,d)のリスト長比較は未完了です。run_lengths.pyを実行してください。')